In [1]:
%reload_ext autoreload
%autoreload 2

In [2]:
import sys
sys.path.append('/rhome/sawale/indus_traning/mlm-fine-tuning/mlm')

In [ ]:
import os
import json
import random
import numpy as np
from transformers import AutoModel, AutoTokenizer, pipeline
import torch
import torch.nn.functional as F
from dotenv import load_dotenv
load_dotenv()
from huggingface_hub import login
login(os.getenv("HUGGINGFACE_TOKEN"))

from preprocess_data import get_dataset


def set_seed(seed=42) -> None:
    """Set all seeds to make results reproducible (deterministic mode).
    When seed is a false-y value or not supplied, disables deterministic mode."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


set_seed(42)

# Load dataset

In [ ]:
config_path = "../config.json"
with open(config_path, "r") as file:
    config = json.load(file)
data_src = "local"
n_rows = None

dataset = get_dataset(config["input"]["dataset"], data_src, n_rows)
print(dataset["test"].shape)

# Load the model and tokenizer

In [ ]:
# Load model or dataset
new_model_name = "nasa-impact/indus-sde-v0.1"
old_model_name = "nasa-impact/nasa-smd-ibm-v0.1"

# Step 1: Load the model and tokenizer
model_names = {"old_model_name": "nasa-impact/nasa-smd-ibm-v0.1", "new_model_name": "nasa-impact/indus-sde-v0.1"}
models = {mn:AutoModel.from_pretrained(hf_model_name) for mn, hf_model_name in model_names.items()}
tokenizers = {mn:AutoTokenizer.from_pretrained(hf_model_name) for mn, hf_model_name in model_names.items()}
embeddings = {mn:None for mn in model_names.keys()}

# Generate embeddings

In [ ]:
from tqdm import tqdm

batch_size = 16  # Adjust based on GPU memory

# Iterate through the dataset in batches
with torch.no_grad():
    for i in tqdm(range(0, len(dataset["test"]), batch_size)):
        # Extract a batch from the dataset by slicing
        batch = dataset["test"][i:i + batch_size]

        # Generate embeddings for each model
        for mn, model in models.items():
            inputs = tokenizers.get(mn)(
                batch["text"],  # Assuming the dataset has a "text" column
                return_tensors="pt",
                padding=True,
                truncation=True,
                max_length=500
            )
            output = model(**inputs)
            # get the last hidden state and aggregrate the embedding per example
            emb = output.last_hidden_state.mean(dim=1, keepdim=False)
            if embeddings.get(mn) is None:
                embeddings[mn] = emb.cpu()
            else:
                embeddings[mn] = torch.cat((embeddings[mn], emb.cpu()), dim=0)
        
torch.save(embeddings, 'model_embeddings.pt')

In [ ]:
# load the embedding if there is embedding file
# check if the file exists first
path = 'model_embeddings.pt'
if os.path.exists(path):
    embeddings = torch.load(path)

In [ ]:
embeddings

# Generate the difference between the embeddings

In [ ]:
embedding_difference = embeddings["new_model_name"] - embeddings["old_model_name"]
print(embedding_difference.shape)

# Use UMAP to reduce the dimensionality of the embeddings

In [ ]:
import torch
import umap
import matplotlib.pyplot as plt
import numpy as np
# Set the random seed


sample_size = 1000

# Initialize UMAP model for dimensionality reduction
umap_model = umap.UMAP(n_components=2)

old_embeddings = embeddings['old_model_name'].numpy()[:sample_size]
new_embeddings = embeddings['new_model_name'].numpy()[:sample_size]

combined_embeddings = np.concatenate((old_embeddings, new_embeddings), axis=0)
umap_model.fit(combined_embeddings)

# Apply UMAP to reduce to 2D
old_embeddings_2d = umap_model.transform(old_embeddings)
new_embeddings_2d = umap_model.transform(new_embeddings)

# Create the plot
plt.figure(figsize=(10, 8))

# Scatter plot for old model embeddings
plt.scatter(old_embeddings_2d[:, 0], old_embeddings_2d[:, 1], c='blue', label='Old Model', alpha=0.6)

# Scatter plot for new model embeddings
plt.scatter(new_embeddings_2d[:, 0], new_embeddings_2d[:, 1], c='red', label='New Model', alpha=0.6)

# Draw lines/arrows from old to new embeddings
for i in range(len(old_embeddings_2d)):
    plt.plot([old_embeddings_2d[i, 0], new_embeddings_2d[i, 0]], 
             [old_embeddings_2d[i, 1], new_embeddings_2d[i, 1]], 
             color='gray', linestyle='-', alpha=0.4)

# Add labels and legend
plt.xlabel('UMAP 1')
plt.ylabel('UMAP 2')
plt.title('Visualization of Old and New Model Embeddings')
plt.legend()

# Show the plot
plt.show()



In [ ]:
import torch
import umap
import matplotlib.pyplot as plt
import numpy as np
from sklearn.decomposition import PCA

# Set the random seed for reproducibility
np.random.seed(42)

# Sample size
sample_size = None

# Retrieve embeddings
old_embeddings = embeddings['old_model_name'].numpy()[:sample_size]
new_embeddings = embeddings['new_model_name'].numpy()[:sample_size]

# Combine embeddings for dimensionality reduction
combined_embeddings = np.concatenate((old_embeddings, new_embeddings), axis=0)

# UMAP dimensionality reduction
umap_model = umap.UMAP(n_components=2)
umap_model.fit(combined_embeddings)
old_embeddings_2d_umap = umap_model.transform(old_embeddings)
new_embeddings_2d_umap = umap_model.transform(new_embeddings)

# PCA dimensionality reduction
pca_model = PCA(n_components=2)
pca_model.fit(combined_embeddings)
old_embeddings_2d_pca = pca_model.transform(old_embeddings)
new_embeddings_2d_pca = pca_model.transform(new_embeddings)

# Create side-by-side plots
fig, axes = plt.subplots(2, 1, figsize=(8, 16))

# UMAP Plot
axes[0].scatter(old_embeddings_2d_umap[:, 0], old_embeddings_2d_umap[:, 1], c='blue', label='Old Model', alpha=0.2)
axes[0].scatter(new_embeddings_2d_umap[:, 0], new_embeddings_2d_umap[:, 1], c='red', label='New Model', alpha=0.2)
for i in range(len(old_embeddings_2d_umap)):
    axes[0].plot([old_embeddings_2d_umap[i, 0], new_embeddings_2d_umap[i, 0]], 
                 [old_embeddings_2d_umap[i, 1], new_embeddings_2d_umap[i, 1]], 
                 color='gray', linestyle='-', alpha=0.02)
axes[0].set_title('UMAP Visualization')
axes[0].set_xlabel('UMAP 1')
axes[0].set_ylabel('UMAP 2')
axes[0].legend()

# PCA Plot
axes[1].scatter(old_embeddings_2d_pca[:, 0], old_embeddings_2d_pca[:, 1], c='blue', label='Old Model', alpha=0.2)
axes[1].scatter(new_embeddings_2d_pca[:, 0], new_embeddings_2d_pca[:, 1], c='red', label='New Model', alpha=0.2)
for i in range(len(old_embeddings_2d_pca)):
    axes[1].plot([old_embeddings_2d_pca[i, 0], new_embeddings_2d_pca[i, 0]], 
                 [old_embeddings_2d_pca[i, 1], new_embeddings_2d_pca[i, 1]], 
                 color='gray', linestyle='-', alpha=0.02)
axes[1].set_title('PCA Visualization')
axes[1].set_xlabel('PCA 1')
axes[1].set_ylabel('PCA 2')
axes[1].legend()

# Show the plots
plt.tight_layout()
plt.show()


In [12]:
# Find the cosine similarity between the embeddings
cosine_similarity = F.cosine_similarity(embeddings["new_model_name"], embeddings["old_model_name"], dim=1)

In [ ]:
# Compute the five-number summary
min_value = torch.min(cosine_similarity)  # Minimum
q1 = torch.quantile(cosine_similarity, 0.25)  # First quartile
median = torch.quantile(cosine_similarity, 0.5)  # Median (Q2)
mean = torch.mean(cosine_similarity)
q3 = torch.quantile(cosine_similarity, 0.75)  # Third quartile
max_value = torch.max(cosine_similarity)  # Maximum
std_value = torch.std(cosine_similarity) 

# Round values to 4 decimal places
min_value = round(min_value.item(), 4)
q1 = round(q1.item(), 4)
median = round(median.item(), 4)
mean = round(mean.item(), 4)
q3 = round(q3.item(), 4)
max_value = round(max_value.item(), 4)
std_value = round(std_value.item(), 4)

# Display the five-number summary
print(f"Minimum: {min_value}")
print(f"First Quartile (Q1): {q1}")
print(f"Median (Q2): {median}")
print(f"Mean: {mean}")
print(f"Third Quartile (Q3): {q3}")
print(f"Maximum: {max_value}")
print(f"Standard Deviation: {std_value}")

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# Convert data to numpy for compatibility with seaborn/matplotlib
data_np = cosine_similarity.numpy()

# Create the figure and axis
plt.figure(figsize=(12, 6))

# Distribution Plot (Histogram + KDE)
plt.subplot(1, 2, 1)  # First subplot
sns.histplot(data_np, kde=True, color='blue', bins=50)
plt.title("Distribution Plot of Cosine Similarity")
plt.xlabel("Cosine Similarity")
plt.ylabel("Frequency")

# Box Plot
plt.subplot(1, 2, 2)  # Second subplot
sns.boxplot(x=data_np, color='green')
plt.title("Box Plot of Cosine Similarity")
plt.xlabel("Cosine Similarity")

# Display the plots
plt.tight_layout()
plt.show()